# OmniBird — **Fully patch-based** (Point-MAE-aligned) on CIFAR10-DVS

Everything patched: dataset, encoder, target, predictor. Matches Point-MAE
/ Point-BERT / i-JEPA convention end-to-end.

| Stage | Tokens |
|-------|--------|
| Raw clip | ~5–8 K events |
| **Patchify** (curve-sort + reshape + mini-PointNet) | 256 patch tokens |
| Encoder (dense self-attn) | 192 context patches → 192 features |
| Target encoder | 256 patches → 256 features |
| Predictor (perceiver, 64 queries) | 64 target patch queries cross-attend to 192 context |
| **Loss** | smooth_L1 on (B, 64, D) — 64 per-patch predictions per sample |

Compute (~25× faster encoder, ~6× faster overall vs per-event).
Target signal is stable per-patch summaries.
Encoder still gives per-patch features for classification probe.


## 1. Setup

In [ ]:
import os, sys, math, time, copy, ssl, glob
sys.path.insert(0, os.path.abspath('..'))
ssl._create_default_https_context = ssl._create_unverified_context

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import matplotlib.pyplot as plt

from omnibird import (
    OmniBirdConfig,
    PatchOmniBirdEncoder, PerceiverPredictor,
    OmniBirdPatchDataset,
    TargetCenter, ema_update, make_momentum_schedule,
    jepa_loss, diag_dict, fmt_diag,
    save_atomic, ensure_dir, short_params, count_params,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0); np.random.seed(0)

N_GPUS_REQUESTED = 4
N_GPUS = min(N_GPUS_REQUESTED, torch.cuda.device_count())
USE_DP = (DEVICE == "cuda") and (N_GPUS > 1)
print(f"GPUs visible = {torch.cuda.device_count()}  using = {N_GPUS}  DataParallel = {USE_DP}")

DATA_ROOT  = "../data/cifar10_dvs_omnibird"
TRAIN_FRAC = 0.8

cfg = OmniBirdConfig()
cfg.coord_dim       = 3
cfg.signal_dim      = 2                # one-hot ON/OFF
cfg.n_events_total  = 0
cfg.n_events_max    = 8192             # P_max * K_per = 256 * 32

# ── PATCH MODE ─────────────────────────────────────────────────────────
cfg.patch_mode         = True
cfg.patch_size         = 32            # K events per patch
cfg.patches_per_block  = 16            # target patches per block
cfg.n_pred_blocks      = 4
cfg.n_tgt              = cfg.n_pred_blocks * cfg.patches_per_block      # = 64 target patches
cfg.ctx_max_patches    = 192
cfg.patch_curve        = "hilbert"
cfg.context_target_margin = 0.0        # margin in patch-centroid space, optional

cfg.side               = 64

# JEPA
cfg.loss_type      = "cosine"
cfg.use_centering  = False
cfg.ema_start      = 0.999
cfg.ema_end        = 0.9999
cfg.probe_patience = 5

# Optim
cfg.epochs         = 100
cfg.batch_size     = 64                # 64/4 = 16 per GPU
cfg.lr             = 8e-4
cfg.probe_interval = 10
cfg.probe_epochs   = 3
cfg.log_every      = 25
cfg.ckpt_dir       = "./checkpoints_omnibird_cifar10_dvs_patch"
ensure_dir(cfg.ckpt_dir)

print(f"P_max={8192 // cfg.patch_size} patches  K={cfg.patch_size} events/patch")
print(f"target patches={cfg.n_tgt}  context_max={cfg.ctx_max_patches}")


## 2. Dataset wrapper (reads CIFAR10-DVS clips)

In [ ]:
class CIFAR10DVSFromClips:
    def __init__(self, root, sensor_hw=(128, 128)):
        self.root = root
        self.h, self.w = sensor_hw
        self.clips = sorted(glob.glob(os.path.join(root, "clip_*")))
        if not self.clips:
            raise RuntimeError(f"No clip_* in {root} (resolved: {os.path.abspath(root)})")
    def __len__(self): return len(self.clips)
    def __getitem__(self, idx):
        clip = self.clips[idx]
        ev = np.load(os.path.join(clip, "events_0.npy")).astype(np.float32)
        if ev.shape[0] == 0:
            ev = np.zeros((1, 4), dtype=np.float32)
        x = ev[:, 0] / max(self.w - 1, 1) * 2.0 - 1.0
        y = ev[:, 1] / max(self.h - 1, 1) * 2.0 - 1.0
        t_raw = ev[:, 2]
        t = (t_raw - t_raw.min()) / max(t_raw.max() - t_raw.min(), 1.0) * 2.0 - 1.0
        p = ev[:, 3]
        if p.max() <= 1.0 and p.min() >= 0.0:
            p = p * 2.0 - 1.0
        out = np.stack([x, y, t, p], axis=1).astype(np.float32)
        with open(os.path.join(clip, "label_0.txt")) as f:
            label = int(f.read().strip())
        return out, label


base = CIFAR10DVSFromClips(DATA_ROOT)
n = len(base); rng = np.random.RandomState(0)
perm = rng.permutation(n); n_train = int(n * TRAIN_FRAC)
train_idx, test_idx = perm[:n_train], perm[n_train:]
class _Subset:
    def __init__(self, b, i): self.b=b; self.i=i
    def __len__(self): return len(self.i)
    def __getitem__(self, k): return self.b[int(self.i[k])]
base_train = _Subset(base, train_idx); base_test = _Subset(base, test_idx)
print(f"loaded CIFAR10-DVS: train={len(base_train)}  test={len(base_test)}")

from torch.utils.data import DataLoader
train_ds      = OmniBirdPatchDataset(base_train, cfg, train=True,
                                       patches_per_block=cfg.patches_per_block,
                                       n_pred_blocks=cfg.n_pred_blocks,
                                       ctx_max=cfg.ctx_max_patches,
                                       patch_size=cfg.patch_size,
                                       margin=cfg.context_target_margin,
                                       curve=cfg.patch_curve)
train_eval_ds = OmniBirdPatchDataset(base_train, cfg, train=False,
                                       ctx_max=cfg.ctx_max_patches,
                                       patch_size=cfg.patch_size,
                                       curve=cfg.patch_curve)
test_ds       = OmniBirdPatchDataset(base_test, cfg, train=False,
                                       ctx_max=cfg.ctx_max_patches,
                                       patch_size=cfg.patch_size,
                                       curve=cfg.patch_curve)

train_loader      = DataLoader(train_ds,      batch_size=cfg.batch_size, shuffle=True,  num_workers=0, pin_memory=False)
train_eval_loader = DataLoader(train_eval_ds, batch_size=cfg.batch_size, shuffle=True,  num_workers=0, pin_memory=False)
test_loader       = DataLoader(test_ds,       batch_size=cfg.batch_size, shuffle=False, num_workers=0, pin_memory=False)

batch = next(iter(train_loader))
for k, v in batch.items():
    if hasattr(v, "shape"):
        print(f"  {k:24s} {tuple(v.shape)}  {v.dtype}")


## 3. Models

In [ ]:
context_encoder = PatchOmniBirdEncoder(
    d_model=cfg.d_model, n_layers=cfg.n_layers_enc,
    n_heads=cfg.n_heads, dim_head=cfg.dim_head, ffn_mult=cfg.ffn_mult,
    signal_dim=cfg.signal_dim, coord_dim=cfg.coord_dim,
    fourier_dim=cfg.fourier_dim, fourier_scale=cfg.fourier_scale,
    patch_size=cfg.patch_size,
).to(DEVICE)
target_encoder = copy.deepcopy(context_encoder).to(DEVICE)
for q in target_encoder.parameters(): q.requires_grad_(False)
predictor = PerceiverPredictor(
    d_model=cfg.d_model, d_pred=cfg.d_pred,
    n_layers=cfg.n_layers_pred,
    n_heads=cfg.n_heads_pred, dim_head=cfg.dim_head_pred,
    coord_dim=cfg.coord_dim,
    fourier_dim=cfg.fourier_dim, fourier_scale=cfg.fourier_scale,
    ffn_mult=cfg.ffn_mult, pos_symmetric=cfg.predictor_pos_symmetric,
).to(DEVICE)
target_center = TargetCenter(cfg.d_model, momentum=0.9).to(DEVICE)

if USE_DP:
    device_ids = list(range(N_GPUS))
    context_encoder = nn.DataParallel(context_encoder, device_ids=device_ids)
    target_encoder  = nn.DataParallel(target_encoder,  device_ids=device_ids)
    predictor       = nn.DataParallel(predictor,       device_ids=device_ids)
    print(f"wrapped in DataParallel on {device_ids}")

def _unwrap(m):
    return m.module if isinstance(m, nn.DataParallel) else m

print(f"context_encoder: {short_params(_unwrap(context_encoder))}  "
      f"predictor (perceiver): {short_params(_unwrap(predictor))}")


## 4. Optimizer + schedules + resume

In [ ]:
params = list(_unwrap(context_encoder).parameters()) + list(_unwrap(predictor).parameters())
optimizer = AdamW(params, lr=cfg.lr, weight_decay=cfg.weight_decay)
total_steps = cfg.epochs * len(train_loader)
scheduler = CosineAnnealingLR(optimizer, T_max=total_steps)
momentum_gen = make_momentum_schedule(cfg.ema_start, cfg.ema_end, total_steps)

PBB_LAST       = os.path.join(cfg.ckpt_dir, "omnibird_last.pt")
PBB_BEST       = os.path.join(cfg.ckpt_dir, "omnibird_best.pt")
PBB_PROBE_BEST = os.path.join(cfg.ckpt_dir, "omnibird_probe_best.pt")

RESUME = True
history = {"loss": [], "diag_steps": [], "diag_log": [], "probe_accs": []}
start_epoch, best_loss, global_step = 1, float("inf"), 0
best_probe_acc, probe_no_improve = -1.0, 0
m = cfg.ema_start

if RESUME and os.path.exists(PBB_LAST):
    s = torch.load(PBB_LAST, map_location=DEVICE, weights_only=False)
    _unwrap(context_encoder).load_state_dict(s["context_encoder"])
    _unwrap(target_encoder).load_state_dict(s["target_encoder"])
    _unwrap(predictor).load_state_dict(s["predictor"])
    target_center.load_state_dict(s["center"])
    optimizer.load_state_dict(s["opt"]); scheduler.load_state_dict(s["sched"])
    history = s.get("history", history)
    global_step = s.get("global_step", 0); best_loss = s.get("best_loss", float("inf"))
    best_probe_acc = s.get("best_probe_acc", -1.0); probe_no_improve = s.get("probe_no_improve", 0)
    start_epoch = s["epoch"] + 1
    for _ in range(global_step):
        try: m = next(momentum_gen)
        except StopIteration: m = cfg.ema_end
    print(f"resumed @ ep {s['epoch']}, step {global_step}, best_probe_acc={best_probe_acc:.4f}")
else:
    print("starting fresh.")


## 5. Patch-level probe (mean-pool over real context patches)

In [ ]:
class LinearProbe(nn.Module):
    def __init__(self, in_dim, n_classes=10):
        super().__init__()
        self.fc = nn.Linear(in_dim, n_classes)
    def forward(self, z): return self.fc(z)


def _gather_ctx_patches(batch):
    pool_ev = batch["pool_patch_events"].to(DEVICE)
    pool_cen = batch["pool_patch_centroids"].to(DEVICE)
    pool_kpm = batch["pool_patch_kpm"].to(DEVICE)
    pool_evkpm = batch["pool_event_kpm"].to(DEVICE)
    ctx_idx = batch["ctx_patch_idx"].to(DEVICE)
    return pool_ev, pool_cen, pool_kpm, pool_evkpm, ctx_idx


def _encode_ctx_patches(encoder, batch):
    pool_ev, pool_cen, pool_kpm, pool_evkpm, ctx_idx = _gather_ctx_patches(batch)
    # Gather context patches from the pool
    B, P, K, F = pool_ev.shape
    Bc, Pc = ctx_idx.shape
    idx_e = ctx_idx.unsqueeze(-1).unsqueeze(-1).expand(Bc, Pc, K, F)
    ctx_ev = torch.gather(pool_ev, 1, idx_e)
    idx_c = ctx_idx.unsqueeze(-1).expand(Bc, Pc, pool_cen.size(-1))
    ctx_cen = torch.gather(pool_cen, 1, idx_c)
    ctx_kpm = torch.gather(pool_kpm, 1, ctx_idx)
    ctx_evkpm = torch.gather(pool_evkpm, 1, ctx_idx.unsqueeze(-1).expand(Bc, Pc, K))
    return ctx_ev, ctx_cen, ctx_kpm, ctx_evkpm


def quick_probe(num_epochs=cfg.probe_epochs, lr=1e-3, wd=1e-4):
    enc = _unwrap(context_encoder)
    saved_train = enc.training
    saved_rg = [p.requires_grad for p in enc.parameters()]
    for p in enc.parameters(): p.requires_grad_(False)
    probe = LinearProbe(cfg.d_model, 10).to(DEVICE)
    opt = AdamW(probe.parameters(), lr=lr, weight_decay=wd)
    ce = nn.CrossEntropyLoss()

    def _z(batch):
        ctx_ev, ctx_cen, ctx_kpm, ctx_evkpm = _encode_ctx_patches(enc, batch)
        with torch.no_grad():
            enc.eval()
            g = enc(ctx_ev, ctx_cen, patch_kpm=ctx_kpm, kpm_per_event=ctx_evkpm)  # (B, Pc, D)
        # Masked mean-pool over real (non-pad) patches
        mask = (~ctx_kpm).unsqueeze(-1).float()
        return (g * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)

    for _ in range(num_epochs):
        probe.train()
        for b in train_eval_loader:
            y = b["label"].to(DEVICE)
            z = _z(b)
            opt.zero_grad(set_to_none=True)
            ce(probe(z), y).backward()
            opt.step()
    probe.eval()
    correct = total = 0
    with torch.no_grad():
        for b in test_loader:
            y = b["label"].to(DEVICE)
            z = _z(b)
            correct += (probe(z).argmax(-1) == y).sum().item(); total += y.size(0)
    for p, rg in zip(enc.parameters(), saved_rg): p.requires_grad_(rg)
    enc.train(saved_train)
    return correct / max(total, 1)


## 6. Training loop

In [ ]:
def _index_patches(batch, key_idx, key_ev, key_cen, key_kpm, key_evkpm):
    # Pull a subset of patches from the pool tensors via key_idx.
    pool_ev   = batch[key_ev].to(DEVICE)
    pool_cen  = batch[key_cen].to(DEVICE)
    pool_kpm  = batch[key_kpm].to(DEVICE)
    pool_evkpm= batch[key_evkpm].to(DEVICE)
    idx       = batch[key_idx].to(DEVICE)
    B, P, K, F = pool_ev.shape
    Bc, Pc = idx.shape
    sub_ev   = torch.gather(pool_ev, 1, idx.unsqueeze(-1).unsqueeze(-1).expand(Bc, Pc, K, F))
    sub_cen  = torch.gather(pool_cen, 1, idx.unsqueeze(-1).expand(Bc, Pc, pool_cen.size(-1)))
    sub_kpm  = torch.gather(pool_kpm, 1, idx)
    sub_evkpm= torch.gather(pool_evkpm, 1, idx.unsqueeze(-1).expand(Bc, Pc, K))
    return sub_ev, sub_cen, sub_kpm, sub_evkpm, idx


epoch = start_epoch - 1
try:
    for epoch in range(start_epoch, cfg.epochs + 1):
        context_encoder.train(); predictor.train()
        epoch_loss, steps = 0.0, 0
        t0 = time.time()

        for batch in train_loader:
            # Ctx patches
            ctx_ev, ctx_cen, ctx_kpm, ctx_evkpm, _ = _index_patches(
                batch, "ctx_patch_idx",
                "pool_patch_events", "pool_patch_centroids",
                "pool_patch_kpm", "pool_event_kpm")
            # Pool (all patches) for target encoder
            pool_ev   = batch["pool_patch_events"].to(DEVICE)
            pool_cen  = batch["pool_patch_centroids"].to(DEVICE)
            pool_kpm  = batch["pool_patch_kpm"].to(DEVICE)
            pool_evkpm= batch["pool_event_kpm"].to(DEVICE)
            # Target patch indices and centroids
            tgt_idx  = batch["tgt_patch_idx"].to(DEVICE)
            tgt_cen  = batch["tgt_patch_centroids"].to(DEVICE)

            with torch.no_grad():
                g_tgt_all = target_encoder(pool_ev, pool_cen,
                                           patch_kpm=pool_kpm,
                                           kpm_per_event=pool_evkpm)        # (B, P, D)
                # Gather target patches
                Bg, Pg, Dg = g_tgt_all.shape
                Bt, Pt = tgt_idx.shape
                h_tgt_raw = torch.gather(g_tgt_all, 1,
                                          tgt_idx.unsqueeze(-1).expand(Bt, Pt, Dg))   # (B, Pt, D)
                if cfg.use_centering:
                    target_center.update(h_tgt_raw)
                    h_tgt = F.layer_norm(target_center(h_tgt_raw), (h_tgt_raw.size(-1),))
                else:
                    h_tgt = F.layer_norm(h_tgt_raw, (h_tgt_raw.size(-1),))

            g_ctx = context_encoder(ctx_ev, ctx_cen,
                                     patch_kpm=ctx_kpm,
                                     kpm_per_event=ctx_evkpm)               # (B, Pc, D)
            # Predictor: queries at target-patch centroids
            h_pred = predictor(g_ctx, tgt_cen,
                                ctx_coords=ctx_cen if cfg.predictor_pos_symmetric else None,
                                ctx_key_padding_mask=ctx_kpm)               # (B, Pt, D)

            loss = jepa_loss(h_pred, h_tgt, loss_type=cfg.loss_type)
            optimizer.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            optimizer.step(); scheduler.step()

            try: m = next(momentum_gen)
            except StopIteration: m = cfg.ema_end
            ema_update(_unwrap(target_encoder), _unwrap(context_encoder), m)

            global_step += 1; epoch_loss += loss.item(); steps += 1
            if global_step % cfg.log_every == 0:
                d = diag_dict(loss, h_pred, h_tgt, g_ctx, target_center)
                print(fmt_diag(d, global_step, epoch, scheduler.get_last_lr()[0], m))
                history["diag_steps"].append(global_step); history["diag_log"].append(d)

        avg = epoch_loss / max(steps, 1)
        history["loss"].append(avg)
        improved = avg < best_loss
        if improved: best_loss = avg

        ckpt_state = {
            "epoch": epoch, "cfg": cfg.__dict__,
            "context_encoder": _unwrap(context_encoder).state_dict(),
            "target_encoder":  _unwrap(target_encoder).state_dict(),
            "predictor":       _unwrap(predictor).state_dict(),
            "center":          target_center.state_dict(),
            "opt": optimizer.state_dict(), "sched": scheduler.state_dict(),
            "global_step": global_step, "best_loss": best_loss,
            "best_probe_acc": best_probe_acc, "probe_no_improve": probe_no_improve,
            "history": history,
        }
        if improved: save_atomic(ckpt_state, PBB_BEST)
        save_atomic(ckpt_state, PBB_LAST)
        print(f"=== ep {epoch:03d}/{cfg.epochs}  avg_loss={avg:.4f}  m={m:.4f}  {time.time()-t0:.1f}s"
              + ("  ★ new best" if improved else ""))

        if cfg.probe_interval > 0 and epoch % cfg.probe_interval == 0:
            t_probe = time.time()
            print(f"  → probe at epoch {epoch}...")
            acc = quick_probe()
            history.setdefault("probe_accs", []).append((epoch, acc))
            print(f"  [probe ep {epoch:03d}]  test_acc = {acc:.4f}  ({time.time()-t_probe:.1f}s)")
            if acc > best_probe_acc:
                best_probe_acc = acc; probe_no_improve = 0
                ckpt_state["best_probe_acc"] = best_probe_acc
                ckpt_state["probe_no_improve"] = probe_no_improve
                ckpt_state["history"] = history
                save_atomic(ckpt_state, PBB_PROBE_BEST)
                print(f"  ★ new best probe acc → {best_probe_acc:.4f}")
            else:
                probe_no_improve += 1
                print(f"  no probe improvement ({probe_no_improve}/{cfg.probe_patience})  best so far = {best_probe_acc:.4f}")
            if cfg.probe_patience > 0 and probe_no_improve >= cfg.probe_patience:
                print(f"\nEarly stopping at epoch {epoch}: best probe acc = {best_probe_acc:.4f}")
                break

    print("\nDone.")
except KeyboardInterrupt:
    print(f"\nInterrupted at epoch {epoch}.  Checkpoints in {cfg.ckpt_dir}.")


## 7. Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
if history["loss"]:
    axes[0].plot(range(1, len(history["loss"])+1), history["loss"], lw=1.5)
    axes[0].set_xlabel("epoch"); axes[0].set_ylabel("avg loss"); axes[0].set_title("JEPA loss"); axes[0].grid(alpha=0.3)
if history["diag_log"]:
    steps = history["diag_steps"]
    cos_mean = [d["cos_mean"] for d in history["diag_log"]]
    axes[1].plot(steps, cos_mean, color='C2'); axes[1].set_ylim(-0.1, 1.05)
    axes[1].set_xlabel("step"); axes[1].set_title("cos(h_pred, h_tgt)"); axes[1].grid(alpha=0.3)
if history.get("probe_accs"):
    eps, accs = zip(*history["probe_accs"])
    axes[2].plot(eps, [a*100 for a in accs], 'o-', color='C3')
    axes[2].set_ylim(0, 100); axes[2].set_xlabel("epoch"); axes[2].set_ylabel("probe acc (%)")
    axes[2].set_title(f"probe acc (best = {max(accs)*100:.2f}%)"); axes[2].grid(alpha=0.3)
plt.tight_layout(); plt.show()
